# 1. Scrape & combine NYC Citi Bike trip data (2013–2019)

**Who worked on this:** Rune Lemmens

Downloads and combines NYC Citi Bike data from **2013 through 2019** only. This includes the two legacy schemas and excludes the larger 2020+ data which does not have gender and age data. Every trip is retained in one plain CSV with standardized columns.

**Data choices:** Jersey City (`JC-`) files are excluded. Duplicate flat/split monthly copies in yearly archives are deduplicated. Downloads are resumable, and assembly checkpoints completed CSV sources so reruns skip already processed data.

In [3]:
import json
import os
import re
import shutil
import tempfile
import time
import xml.etree.ElementTree as ET
from contextlib import contextmanager
from pathlib import Path
from zipfile import BadZipFile, ZipFile

import pandas as pd
import requests

# --- project scope ---
START_YEAR = 2013
END_YEAR = 2019

# --- paths ---
DATA_DIR = Path.cwd() / "data"
RAW_DIR = DATA_DIR / "raw"
OUTPUT_PATH = DATA_DIR / "processed" / "citibike_trips_unified_2013_2019.csv"
TEMP_OUTPUT_PATH = OUTPUT_PATH.with_suffix(OUTPUT_PATH.suffix + ".tmp")
ASSEMBLY_STATE_PATH = OUTPUT_PATH.with_suffix(".manifest.json")
MEMBER_TEMP_PATH = OUTPUT_PATH.with_suffix(".member.tmp")

# --- download settings ---
BUCKET_URL = "https://s3.amazonaws.com/tripdata/"
DATE_CODE_RE = re.compile(r"^(\d{4})(\d{2})?-citibike-tripdata(?:\.csv)?\.zip$")
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 5
REQUEST_TIMEOUT = (10, 120)

# --- assembly settings ---
CHUNK_SIZE = 200_000
PROGRESS_INTERVAL_SECONDS = 3
MONTH_KEY_RE = re.compile(r"(\d{6})-citibike-tripdata")
UNIFIED_COLUMNS = [
    "started_at", "ended_at",
    "start_station_id", "start_station_name", "start_lat", "start_lng",
    "end_station_id", "end_station_name", "end_lat", "end_lng",
    "member_casual", "birth_year", "gender", "trip_duration_s",
]
OLD_SCHEMA_RENAME = {
    "starttime": "started_at", "start_time": "started_at",
    "stoptime": "ended_at", "stop_time": "ended_at",
    "start_station_id": "start_station_id", "start_station_name": "start_station_name",
    "start_station_latitude": "start_lat", "start_station_longitude": "start_lng",
    "end_station_id": "end_station_id", "end_station_name": "end_station_name",
    "end_station_latitude": "end_lat", "end_station_longitude": "end_lng",
    "birth_year": "birth_year", "gender": "gender",
}
OLD_SCHEMA_MARKERS = ("starttime", "start_time")
USERTYPE_TO_MEMBER_CASUAL = {"subscriber": "member", "customer": "casual"}

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## 2. Download (2013–2019)

Downloads the NYC-system archives for 2013 through 2019. Complete files are skipped and partial files can resume. Post-2019 archives are not included in this project scope.

In [4]:
def list_main_system_files() -> list[tuple[str, int]]:
    """Return (key, size_in_bytes) for all in-scope main-system trip archives."""
    files = []
    continuation_token = None

    while True:
        params = {"list-type": "2"}
        if continuation_token:
            params["continuation-token"] = continuation_token
        response = requests.get(BUCKET_URL, params=params, timeout=30)
        response.raise_for_status()
        root = ET.fromstring(response.content)

        for contents in root.findall("s3:Contents", NS):
            key = contents.findtext("s3:Key", namespaces=NS)
            size = contents.findtext("s3:Size", namespaces=NS)
            if not key or not size or key.startswith("JC-"):
                continue

            match = DATE_CODE_RE.match(key)
            if not match:
                continue
            year = int(match.group(1))
            if year < START_YEAR or (END_YEAR is not None and year > END_YEAR):
                continue
            files.append((key, int(size)))

        continuation_token = root.findtext("s3:NextContinuationToken", namespaces=NS)
        if not continuation_token:
            break

    return sorted(files)


def download_file(key: str, expected_size: int) -> bool:
    """Stream-download a single file with resume + retry. Returns True on success."""
    destination = RAW_DIR / key
    if destination.exists() and destination.stat().st_size == expected_size:
        print(f"[skip] {key} already downloaded")
        return True

    for attempt in range(1, MAX_RETRIES + 1):
        current_size = destination.stat().st_size if destination.exists() else 0
        resume_from = current_size if 0 < current_size < expected_size else 0
        headers = {"Range": f"bytes={resume_from}-"} if resume_from else {}

        try:
            suffix = f" resuming from {resume_from / 1_048_576:,.1f} MB" if resume_from else ""
            print(f"[download] {key} ({expected_size / 1_048_576:,.1f} MB){suffix}")
            with requests.get(BUCKET_URL + key, stream=True, timeout=REQUEST_TIMEOUT, headers=headers) as response:
                response.raise_for_status()
                mode = "ab" if (resume_from and response.status_code == 206) else "wb"
                with open(destination, mode) as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)

            if destination.stat().st_size == expected_size:
                return True
            print(f"[warn] {key} size mismatch, retrying")
        except requests.exceptions.RequestException as exc:
            print(f"[warn] {key} failed on attempt {attempt}/{MAX_RETRIES}: {exc}")
        except OSError as exc:
            print(f"[warn] {key} file access issue on attempt {attempt}/{MAX_RETRIES}: {exc}")

        time.sleep(RETRY_BACKOFF_SECONDS * attempt)

    print(f"[error] giving up on {key} after {MAX_RETRIES} attempts")
    return False

In [5]:
files_to_fetch = list_main_system_files()
total_gb = sum(size for _, size in files_to_fetch) / 1_000_000_000
print(f"Found {len(files_to_fetch):,} archives to download (~{total_gb:,.2f} GB compressed)\n")

failed_downloads = [key for key, size in files_to_fetch if not download_file(key, size)]

print("\nDone downloading. Raw zip files are in:", RAW_DIR)
if failed_downloads:
    print(f"{len(failed_downloads)} file(s) failed, rerun this cell to retry them: {failed_downloads}")

Found 7 archives to download (~4.02 GB compressed)

[skip] 2013-citibike-tripdata.zip already downloaded
[skip] 2014-citibike-tripdata.zip already downloaded
[skip] 2015-citibike-tripdata.zip already downloaded
[skip] 2016-citibike-tripdata.zip already downloaded
[skip] 2017-citibike-tripdata.zip already downloaded
[skip] 2018-citibike-tripdata.zip already downloaded
[skip] 2019-citibike-tripdata.zip already downloaded

Done downloading. Raw zip files are in: c:\Users\lemme\OneDrive\Desktop\2025_School\deep learning\Ctrl-Alt-Defeat\NYC Citi Bike System Data\data\raw


### Downloaded source schemas (2013–2019)

Scans CSV headers in the downloaded 2013–2019 archives and reports distinct schemas and their date ranges.

In [6]:
from collections import defaultdict


def _is_csv_member(name: str) -> bool:
    base = Path(name).name
    return base.lower().endswith(".csv") and "__MACOSX" not in name and not base.startswith("._") and base != ".DS_Store"


def _nested_zip_csv_info(source_zip: ZipFile, nested_zip_name: str) -> list[tuple[str, int, int]]:
    """Read CSV names, sizes, and CRCs from a monthly ZIP nested in a yearly archive."""
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_file:
        temp_path = Path(temp_file.name)
        with source_zip.open(nested_zip_name) as nested_stream:
            shutil.copyfileobj(nested_stream, temp_file)
    try:
        with ZipFile(temp_path) as nested_zip:
            return [
                (info.filename, info.file_size, info.CRC)
                for info in nested_zip.infolist()
                if _is_csv_member(info.filename)
            ]
    finally:
        temp_path.unlink(missing_ok=True)


def discover_source_csvs(source_zip: ZipFile) -> list[tuple[str, str | None]]:
    """List CSVs directly or within one nested ZIP layer, preferring flat monthly copies."""
    grouped_sources = defaultdict(list)
    for info in source_zip.infolist():
        name = info.filename
        if _is_csv_member(name):
            base = Path(name).name
            month_match = MONTH_KEY_RE.search(base)
            month_key = month_match.group(1) if month_match else base
            grouped_sources[month_key].append(((name, None), base, True))
        elif Path(name).name.lower().endswith(".zip") and "__MACOSX" not in name and not Path(name).name.startswith("._"):
            month_match = MONTH_KEY_RE.search(Path(name).name)
            nested_month = month_match.group(1) if month_match else None
            for inner_name, _, _ in _nested_zip_csv_info(source_zip, name):
                inner_match = MONTH_KEY_RE.search(Path(inner_name).name)
                month_key = inner_match.group(1) if inner_match else nested_month or Path(inner_name).name
                grouped_sources[month_key].append(((name, inner_name), Path(inner_name).name, False))

    chosen = []
    for month_key, candidates in grouped_sources.items():
        flat = [item for item in candidates if item[2] and item[1] == f"{month_key}-citibike-tripdata.csv"]
        direct = [item for item in candidates if item[2]]
        selected = flat if flat else direct if direct else candidates
        chosen.extend(descriptor for descriptor, _, _ in selected)
    return chosen


@contextmanager
def open_source_csv(archive_path: Path, descriptor: tuple[str, str | None]):
    """Yield a CSV stream from a top-level archive or a CSV nested in a monthly ZIP."""
    outer_member, inner_member = descriptor
    with ZipFile(archive_path) as source_zip:
        if inner_member is None:
            with source_zip.open(outer_member) as csv_stream:
                yield csv_stream
            return

        with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_file:
            temp_path = Path(temp_file.name)
            with source_zip.open(outer_member) as nested_stream:
                shutil.copyfileobj(nested_stream, temp_file)
        try:
            with ZipFile(temp_path) as nested_zip, nested_zip.open(inner_member) as csv_stream:
                yield csv_stream
        finally:
            temp_path.unlink(missing_ok=True)


schema_by_signature = {}
schemas_by_year = defaultdict(set)
schema_periods = defaultdict(set)
missing_schema_archives = []

for archive_key, expected_size in files_to_fetch:
    archive_path = RAW_DIR / archive_key
    if not archive_path.exists() or archive_path.stat().st_size != expected_size:
        missing_schema_archives.append(archive_key)
        continue

    archive_year = int(DATE_CODE_RE.match(archive_key).group(1))
    with ZipFile(archive_path) as source_zip:
        source_descriptors = discover_source_csvs(source_zip)

    for descriptor in source_descriptors:
        outer_member, inner_member = descriptor
        member_name = inner_member or outer_member
        with open_source_csv(archive_path, descriptor) as source_csv:
            header = pd.read_csv(source_csv, nrows=0)
        normalized_header = tuple(
            column.strip().lower().replace(" ", "_")
            for column in header.columns
        )
        schema_by_signature.setdefault(normalized_header, len(schema_by_signature) + 1)
        schema_id = schema_by_signature[normalized_header]

        month_match = MONTH_KEY_RE.search(Path(member_name).name)
        if month_match:
            year_month = month_match.group(1)
            member_year = int(year_month[:4])
            month_number = int(year_month[4:])
            period_start = f"{member_year:04d}-{month_number:02d}"
            period_end = period_start
        else:
            member_year = archive_year
            period_start = f"{member_year:04d}-01"
            period_end = f"{member_year:04d}-12"

        schemas_by_year[member_year].add(schema_id)
        schema_periods[schema_id].add((period_start, period_end))


def format_schema_ranges(periods: set[tuple[str, str]]) -> str:
    """Combine adjacent monthly/yearly source periods into readable date ranges."""
    ordered_periods = sorted(periods)
    if not ordered_periods:
        return "no date range found"

    ranges = []
    range_start, range_end = ordered_periods[0]
    for next_start, next_end in ordered_periods[1:]:
        end_year, end_month = map(int, range_end.split("-"))
        next_year, next_month = map(int, next_start.split("-"))
        next_after_end = (next_year * 12 + next_month) == (end_year * 12 + end_month + 1)
        if next_after_end:
            range_end = next_end
        else:
            ranges.append((range_start, range_end))
            range_start, range_end = next_start, next_end
    ranges.append((range_start, range_end))
    return ", ".join(start if start == end else f"{start} to {end}" for start, end in ranges)


print("SCHEMA(S) FOUND IN EACH YEAR:")
for year in sorted(schemas_by_year):
    schema_labels = ", ".join(f"Schema {schema_id}" for schema_id in sorted(schemas_by_year[year]))
    print(f"  {year}: {schema_labels}")

print("\nDISTINCT SCHEMAS AND SOURCE DATE RANGES:")
for signature, schema_id in sorted(schema_by_signature.items(), key=lambda item: item[1]):
    print(f"\nSchema {schema_id} — used: {format_schema_ranges(schema_periods[schema_id])}")
    print("  Columns: " + ", ".join(signature))

if missing_schema_archives:
    print(f"\nSkipped {len(missing_schema_archives)} archive(s) that are not fully downloaded yet:")
    for archive_key in missing_schema_archives:
        print(f"  {archive_key}")

SCHEMA(S) FOUND IN EACH YEAR:
  2013: Schema 1
  2014: Schema 1
  2015: Schema 1
  2016: Schema 1, Schema 2
  2017: Schema 1, Schema 2
  2018: Schema 1
  2019: Schema 1

DISTINCT SCHEMAS AND SOURCE DATE RANGES:

Schema 1 — used: 2013-06 to 2016-09, 2017-04 to 2019-12
  Columns: tripduration, starttime, stoptime, start_station_id, start_station_name, start_station_latitude, start_station_longitude, end_station_id, end_station_name, end_station_latitude, end_station_longitude, bikeid, usertype, birth_year, gender

Schema 2 — used: 2016-10 to 2017-03
  Columns: trip_duration, start_time, stop_time, start_station_id, start_station_name, start_station_latitude, start_station_longitude, end_station_id, end_station_name, end_station_latitude, end_station_longitude, bike_id, user_type, birth_year, gender
